# 🦟 Notebook 01 — Carga, Inspección y Limpieza de Datos

**Proyecto:** Dengue en Argentina — EDA (2018–2025)  
**Autora:** Fabiana Grisel González  
**Fuente:** [Portal de Datos Abiertos - Ministerio de Salud](https://datos.salud.gob.ar/dataset/vigilancia-de-dengue-y-zika)  

---

## Objetivo de este notebook

Cargar los datos crudos, realizar una inspección exhaustiva de calidad (con enfoque QA) y generar un dataset limpio listo para el análisis exploratorio.

### Checklist de calidad (QA approach)

- [ ] Verificar tipos de datos
- [ ] Detectar y cuantificar valores nulos
- [ ] Identificar filas duplicadas
- [ ] Validar rangos de fechas
- [ ] Normalizar nombres de jurisdicciones
- [ ] Documentar cada decisión de limpieza

## 1. Configuración del entorno

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

# Agregar src/ al path para importar utils
sys.path.append(str(Path.cwd().parent / 'src'))
from utils import reporte_calidad, configurar_graficos

# Configuración
configurar_graficos()
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print(f'Pandas: {pd.__version__}')
print(f'NumPy: {np.__version__}')

## 2. Carga de datos crudos

Los archivos CSV se descargan desde el [portal de datos abiertos](https://datos.salud.gob.ar/dataset/vigilancia-de-dengue-y-zika) y se guardan en `data/raw/`.

> **Nota:** Si el encoding por defecto (`utf-8`) falla, probar con `latin-1` o `ISO-8859-1`.

In [ ]:
import csv

# ============================================================
# AJUSTAR el nombre del archivo según lo que descargaste
# ============================================================
ARCHIVO_RAW = '../data/raw/dengue_zika_2024_2025.csv'

# Intentar cargar con utf-8; si falla, probar latin-1
try:
    df_raw = pd.read_csv('../data/raw/informacion-publica-dengue-zika-nacional-hasta-20181231.csv')
except UnicodeDecodeError:
    print('⚠️ Error con UTF-8, intentando con latin-1...')
    df_raw = pd.read_csv('../data/raw/informacion-publica-dengue-zika-nacional-hasta-20181231.csv')

print(f'✅ Dataset cargado exitosamente.')
print(f'   Dimensiones: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas')

In [ ]:
# Primeras filas para entender la estructura
df_raw.head(10)

In [ ]:
# Nombres de columnas
print('Columnas del dataset:')
for i, col in enumerate(df_raw.columns):
    print(f'  {i}: {col}')

## 3. Inspección de calidad de datos (QA)

Aplicamos una inspección sistemática como si fuera un **test plan de calidad de datos**.

### 3.1 Tipos de datos e información general

In [ ]:
# Información del DataFrame
df_raw.info()

In [ ]:
# Reporte de calidad personalizado
reporte = reporte_calidad(df_raw)
reporte

### 3.2 Estadísticas descriptivas

In [ ]:
# Estadísticas de columnas numéricas
df_raw.describe()

In [ ]:
# Estadísticas de columnas categóricas
df_raw.describe(include=['object', 'str'])

### 3.3 Análisis de valores nulos

In [ ]:
# Visualización de nulls
nulls = df_raw.isnull().sum()
nulls_pct = (nulls / len(df_raw) * 100).round(2)

null_report = pd.DataFrame({
    'nulls': nulls,
    'porcentaje': nulls_pct
}).sort_values('nulls', ascending=False)

# Mostrar solo columnas con nulls
cols_con_nulls = null_report[null_report['nulls'] > 0]
if len(cols_con_nulls) > 0:
    print(f'⚠️ {len(cols_con_nulls)} columnas con valores nulos:\n')
    print(cols_con_nulls)
else:
    print('✅ No hay valores nulos en el dataset.')

In [ ]:
# Heatmap de nulos (si existen)
if df_raw.isnull().sum().sum() > 0:
    fig, ax = plt.subplots(figsize=(14, 4))
    sns.heatmap(df_raw.isnull().T, cbar=True, cmap='YlOrRd', ax=ax)
    ax.set_title('Mapa de valores nulos', fontweight='bold')
    ax.set_xlabel('Registros')
    ax.set_ylabel('Columnas')
    plt.tight_layout()
    plt.show()
else:
    print('✅ Sin nulos — no se genera heatmap.')

### 3.4 Duplicados

In [ ]:
duplicados = df_raw.duplicated().sum()
print(f'Filas duplicadas: {duplicados:,} de {len(df_raw):,} ({duplicados/len(df_raw)*100:.2f}%)')

if duplicados > 0:
    print('\n⚠️ Ejemplos de filas duplicadas:')
    display(df_raw[df_raw.duplicated(keep=False)].head(10))

### 3.5 Valores únicos por columna categórica

Verificar consistencia en nombres de jurisdicciones y categorías.

In [ ]:
# Listar valores únicos de columnas categóricas (object)
for col in df_raw.select_dtypes(include='object').columns:
    unicos = df_raw[col].nunique()
    print(f'\n📌 {col} ({unicos} valores únicos):')
    if unicos <= 30:
        print(f'   {sorted(df_raw[col].dropna().unique())}')
    else:
        print(f'   (Demasiados para listar — mostrando top 10)')
        print(f'   {df_raw[col].value_counts().head(10).to_dict()}')

## 4. Limpieza de datos

Cada paso de limpieza se documenta con:
- **Problema detectado**
- **Decisión tomada**
- **Impacto** (cuántos registros afectados)

In [ ]:
# Crear copia para limpieza
df = df_raw.copy()
print(f'Dataset copiado para limpieza: {df.shape[0]:,} filas × {df.shape[1]} columnas')

### 4.1 Renombrar columnas (si es necesario)

Normalizamos los nombres a snake_case para consistencia.

In [ ]:
# Normalizar nombres de columnas a snake_case
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('.', '', regex=False)
    .str.replace('á', 'a', regex=False)
    .str.replace('é', 'e', regex=False)
    .str.replace('í', 'i', regex=False)
    .str.replace('ó', 'o', regex=False)
    .str.replace('ú', 'u', regex=False)
    .str.replace('ñ', 'n', regex=False)
)

print('Columnas normalizadas:')
for col in df.columns:
    print(f'  → {col}')

### 4.2 Parseo de fechas

In [ ]:
# ============================================================
# AJUSTAR: Reemplazar 'nombre_columna_fecha' con la columna
# real de fecha del dataset.
# ============================================================

# Ejemplo:
# df['fecha'] = pd.to_datetime(df['nombre_columna_fecha'], errors='coerce')
# errores_fecha = df['fecha'].isnull().sum()
# print(f'Fechas no parseadas: {errores_fecha}')

print('⚠️ PENDIENTE: Ajustar el parseo de fechas según las columnas reales del dataset.')

### 4.3 Eliminar duplicados

In [ ]:
filas_antes = len(df)
df = df.drop_duplicates()
filas_despues = len(df)
eliminadas = filas_antes - filas_despues

print(f'Duplicados eliminados: {eliminadas:,}')
print(f'Dataset: {filas_antes:,} → {filas_despues:,} filas')

### 4.4 Manejo de valores nulos

In [ ]:
# ============================================================
# AJUSTAR según los nulls detectados en la sección 3.3
# ============================================================

# Estrategias posibles:
# 1. Eliminar filas con nulls en columnas críticas
# 2. Rellenar con 'Sin dato' para categóricas
# 3. Rellenar con mediana para numéricas

# Ejemplo:
# df['columna_categorica'] = df['columna_categorica'].fillna('Sin dato')
# df = df.dropna(subset=['columna_critica'])

print('⚠️ PENDIENTE: Definir estrategia de manejo de nulls según el dataset.')

### 4.5 Normalizar jurisdicciones

In [ ]:
# ============================================================
# AJUSTAR: nombre de columna de jurisdicción/provincia
# ============================================================

# Ejemplo:
# df['jurisdiccion'] = df['jurisdiccion'].str.strip().str.title()
# 
# Verificar y corregir inconsistencias:
# print(df['jurisdiccion'].value_counts())

print('⚠️ PENDIENTE: Normalizar nombres de jurisdicciones.')

## 5. Validación post-limpieza

Verificamos que el dataset limpio cumpla con los criterios de calidad.

In [ ]:
print('=' * 60)
print('✅ VALIDACIÓN POST-LIMPIEZA')
print('=' * 60)
print(f'   Filas: {df.shape[0]:,}')
print(f'   Columnas: {df.shape[1]}')
print(f'   Nulls totales: {df.isnull().sum().sum():,}')
print(f'   Duplicados: {df.duplicated().sum():,}')
print(f'   Tipos de datos:\n{df.dtypes.value_counts().to_string()}')
print('=' * 60)

## 6. Exportar dataset limpio

In [ ]:
# Guardar dataset limpio
ARCHIVO_LIMPIO = '../data/processed/dengue_limpio.csv'
df.to_csv(ARCHIVO_LIMPIO, index=False, encoding='utf-8')
print(f'💾 Dataset limpio guardado en: {ARCHIVO_LIMPIO}')
print(f'   Dimensiones finales: {df.shape[0]:,} filas × {df.shape[1]} columnas')

## 📝 Log de decisiones de limpieza

| # | Problema detectado | Decisión | Registros afectados |
|---|-------------------|----------|--------------------|
| 1 | (completar) | (completar) | (completar) |
| 2 | (completar) | (completar) | (completar) |
| 3 | (completar) | (completar) | (completar) |

---
